In [1]:
import pandas as pd

# 파일의 정확한 경로와 확장자(.xlsx)를 지정합니다.
file_path = r'F:\Download\Rawdata.xlsx'

try:
    # 엑셀 파일 로딩 (openpyxl 엔진 사용)
    df = pd.read_excel(file_path, engine='openpyxl')
    print("✅ 엑셀 파일을 성공적으로 불러왔습니다.\n")
    
    # 상위 5개 행 출력
    print(df.head())
    
except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다. '{file_path}' 경로를 확인해주세요.")
except ImportError:
    print("❌ 엑셀 파일을 읽기 위한 'openpyxl' 라이브러리가 설치되어 있지 않습니다.")
    print("💡 터미널(또는 명령 프롬프트)에서 'pip install openpyxl'을 입력하여 설치해주세요.")
except Exception as e:
    print(f"❌ 오류가 발생했습니다: {e}")

✅ 엑셀 파일을 성공적으로 불러왔습니다.

  patient_id 사고 및 질환 항목  연령대  source_count Gender
0    P000001  악성신생물 (암)  10대             1      M
1    P000002  악성신생물 (암)  10대             1      M
2    P000003  악성신생물 (암)  10대             1      M
3    P000004  악성신생물 (암)  10대             1      M
4    P000005  악성신생물 (암)  10대             1      M


In [3]:
import pandas as pd

# 1. Step3 보험 데이터 파일 경로 지정 (.csv)
file_path_step3 = r'F:\Download\Step3_통합_보험데이터_최종_1개월_월환산완료.csv'

try:
    # CSV 파일 로딩 (한글 깨짐 방지를 위해 cp949 인코딩 옵션 추가)
    try:
        df_step3 = pd.read_csv(file_path_step3)
    except UnicodeDecodeError:
        df_step3 = pd.read_csv(file_path_step3, encoding='cp949')
        
    print("✅ Step3 보험 데이터를 성공적으로 불러왔습니다.\n")
    
    # 상위 5개 행 및 전체 보장 컬럼명 출력
    print("📊 [데이터 미리보기]")
    print(df_step3.head())
    
    print("\n🔍 [존재하는 보험 보장 컬럼명 리스트]")
    for col in df_step3.columns:
        print(f" - {col}")
        
except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다. '{file_path_step3}' 경로를 확인해주세요.")
except Exception as e:
    print(f"❌ 오류가 발생했습니다: {e}")

✅ Step3 보험 데이터를 성공적으로 불러왔습니다.

📊 [데이터 미리보기]
        국가     상품플랜  가입나이  성별  최종보험료  3대비급여_MRI/MRA  3대비급여_도수·체외충격파·증식치료  \
0  한국(KRW)  삼성_국내여행    10  남성  12780            0.0                  0.0   
1  한국(KRW)  삼성_국내여행    20  남성  14010            0.0                  0.0   
2  한국(KRW)  삼성_국내여행    30  남성  15360            0.0                  0.0   
3  한국(KRW)  삼성_국내여행    40  남성  16840            0.0                  0.0   
4  한국(KRW)  삼성_국내여행    50  남성  18470            0.0                  0.0   

   3대비급여_비급여주사료  3대비급여_통합한도   사망및장해_상해사망  ...  입원의료비_질병급여(국내)  입원의료비_질병입원  \
0           0.0         0.0  100000000.0  ...             0.0         0.0   
1           0.0         0.0  100000000.0  ...             0.0         0.0   
2           0.0         0.0  100000000.0  ...             0.0         0.0   
3           0.0         0.0  100000000.0  ...             0.0         0.0   
4           0.0         0.0  100000000.0  ...             0.0         0.0   

   통원의료비_상해외래  통원의료비_상해처방조제  통원의료비_질

In [4]:
import pandas as pd

# 1. 119 데이터 파일 경로 지정 (이전에 분류 작업을 마친 파일)
file_path_raw = r'F:\Download\Rawdata_Classified.xlsx'

# 2. 매핑 함수 정의
def map_accident_to_insurance_columns(item):
    if pd.isna(item) or not isinstance(item, str):
        return []

    # Step3 보험 보장 컬럼명
    col_disease_death = '사망및장해_질병사망및80%이상후유장해'
    col_injury_death = '사망및장해_상해사망'
    col_injury_disability = '사망및장해_상해후유장해'
    
    col_disease_inpatient = '입원의료비_질병입원'
    col_disease_outpatient = '통원의료비_질병외래'
    col_injury_inpatient = '입원의료비_상해입원'
    col_injury_outpatient = '통원의료비_상해외래'
    
    col_non_benefit_mri = '3대비급여_MRI/MRA'
    col_non_benefit_manual = '3대비급여_도수·체외충격파·증식치료'
    col_non_benefit_injection = '3대비급여_비급여주사료'
    
    col_overseas_disease = '특수리스크_해외질병의료비(USD)'
    col_overseas_injury = '특수리스크_해외상해의료비(USD)'
    col_rescue = '특수리스크_구원자비용'
    
    # 중증 질환
    critical_diseases = ['악성신생물 (암)', '급성 심장 질환', '뇌혈관 질환']
    if item in critical_diseases:
        return [col_disease_death, col_disease_inpatient, col_overseas_disease]
        
    # 중대 상해
    severe_injuries = ['운수 사고 (교통사고)', '불의의 물에 빠짐 (익사)', '연기, 불 및 불꽃에 노출']
    if item in severe_injuries:
        return [col_injury_death, col_injury_disability, col_injury_inpatient, col_overseas_injury, col_rescue]
        
    # 근골격계 외상
    elif item == '추락 (미끄러짐)':
        return [col_injury_outpatient, col_non_benefit_manual, col_non_benefit_mri, col_non_benefit_injection]
        
    # 일반 외상
    elif item in ['가해 (타살)', '유독성 물질에 의한 불의의 중독 및 노출']:
        return [col_injury_outpatient, col_injury_inpatient, col_overseas_injury]
        
    # 면책
    elif item in ['고의적 자해 (자살)', '고의적 자해(자살)', '정신활성 물질 사용에 의한 정신 및 행동 장애']:
        return []
        
    # 그 외 일반 질환
    else:
        return [col_disease_outpatient, col_disease_inpatient, col_overseas_disease]

try:
    print("⏳ 119 데이터를 불러오는 중입니다...")
    df_raw = pd.read_excel(file_path_raw, engine='openpyxl')
    print("✅ 119 데이터를 성공적으로 불러왔습니다.\n")
    
    print("⚙️ 사고 항목에 실제 보험 보장 컬럼들을 매칭하는 중입니다...")
    # 함수 적용 (리스트 형태 반환)
    df_raw['필요_보장_컬럼_리스트'] = df_raw['사고 및 질환 항목'].apply(map_accident_to_insurance_columns)
    
    # 엑셀 저장 및 가독성을 위해 문자열(Text)로 변환한 컬럼 추가
    df_raw['필요_보장_컬럼(Text)'] = df_raw['필요_보장_컬럼_리스트'].apply(lambda x: ', '.join(x) if isinstance(x, list) else '')
    
    print("✅ 매핑 작업이 완료되었습니다. (데이터프레임에 임시 저장됨)")

except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다. '{file_path_raw}' 경로를 확인해주세요.")
except Exception as e:
    print(f"❌ 오류가 발생했습니다: {e}")

⏳ 119 데이터를 불러오는 중입니다...
✅ 119 데이터를 성공적으로 불러왔습니다.

⚙️ 사고 항목에 실제 보험 보장 컬럼들을 매칭하는 중입니다...
✅ 매핑 작업이 완료되었습니다. (데이터프레임에 임시 저장됨)


In [8]:
# 1. 저장할 새로운 파일 경로 지정
output_file_path = r'F:\Download\Step5_Rawdata_Mapped_to_Columns.xlsx'

try:
    print("📊 [사고별 매칭된 보험 컬럼 샘플 10개 확인]")
    # 중복된 사고 항목을 제외하고 샘플 출력
    preview = df_raw[['사고 및 질환 항목', '필요_보장_컬럼(Text)']].drop_duplicates().head(10)
    for index, row in preview.iterrows():
        print(f"🚨 {row['사고 및 질환 항목']}")
        print(f" ➡️ {row['필요_보장_컬럼(Text)']}\n")
        
    print("💾 매핑이 완료된 최종 데이터를 엑셀로 저장합니다...")
    # 리스트 객체가 들어있는 '필요_보장_컬럼_리스트'는 엑셀 저장 시 오류를 낼 수 있으므로 제외하고 저장
    df_to_save = df_raw.drop(columns=['필요_보장_컬럼_리스트'])
    
    df_to_save.to_excel(output_file_path, index=False, engine='openpyxl')
    print(f"\n🎉 모든 과정이 완료되었습니다! 파일이 저장되었습니다:\n 👉 {output_file_path}")

except NameError:
    print("❌ 'df_raw' 데이터를 찾을 수 없습니다. 반드시 Step 2 코드를 먼저 실행해주세요.")
except Exception as e:
    print(f"❌ 오류가 발생했습니다: {e}")

📊 [사고별 매칭된 보험 컬럼 샘플 10개 확인]
🚨 악성신생물 (암)
 ➡️ 사망및장해_질병사망및80%이상후유장해, 입원의료비_질병입원, 특수리스크_해외질병의료비(USD)

🚨 심장 질환
 ➡️ 통원의료비_질병외래, 입원의료비_질병입원, 특수리스크_해외질병의료비(USD)

🚨 폐렴
 ➡️ 통원의료비_질병외래, 입원의료비_질병입원, 특수리스크_해외질병의료비(USD)

🚨 뇌혈관 질환
 ➡️ 사망및장해_질병사망및80%이상후유장해, 입원의료비_질병입원, 특수리스크_해외질병의료비(USD)

🚨 고의적 자해(자살)
 ➡️ 

🚨 당뇨병
 ➡️ 통원의료비_질병외래, 입원의료비_질병입원, 특수리스크_해외질병의료비(USD)

🚨 간의 질환
 ➡️ 통원의료비_질병외래, 입원의료비_질병입원, 특수리스크_해외질병의료비(USD)

🚨 출생전후기에 기원한 특정 병태
 ➡️ 통원의료비_질병외래, 입원의료비_질병입원, 특수리스크_해외질병의료비(USD)

🚨 선천 기형 / 영아 급사 증후군
 ➡️ 통원의료비_질병외래, 입원의료비_질병입원, 특수리스크_해외질병의료비(USD)

🚨 영아 급사 증후군
 ➡️ 통원의료비_질병외래, 입원의료비_질병입원, 특수리스크_해외질병의료비(USD)

💾 매핑이 완료된 최종 데이터를 엑셀로 저장합니다...

🎉 모든 과정이 완료되었습니다! 파일이 저장되었습니다:
 👉 F:\Download\Step5_Rawdata_Mapped_to_Columns.xlsx


In [9]:
import pandas as pd

# 1. 파일 경로 지정 (기존 분류된 119 데이터 사용)
input_file = r'F:\Download\Rawdata_Classified.xlsx'
output_file = r'F:\Download\Step5_Age_Weight.xlsx'

try:
    print("⏳ 1. 연령대별 데이터를 분석 중입니다...")
    df = pd.read_excel(input_file, engine='openpyxl')
    
    # 연령대별 발생 건수 집계
    age_counts = df['연령대'].value_counts().reset_index()
    age_counts.columns = ['연령대', '발생건수']
    
    # 가중치 계산 (가장 사고가 많은 연령대를 1.0으로 설정)
    max_count = age_counts['발생건수'].max()
    age_counts['Age_Weight'] = (age_counts['발생건수'] / max_count).round(4)
    
    # 결과 출력 및 저장
    print("\n📊 [연령대별 가중치 산출 결과]")
    print(age_counts.to_string(index=False))
    
    age_counts.to_excel(output_file, index=False, engine='openpyxl')
    print(f"\n💾 저장 완료: {output_file}\n")

except FileNotFoundError:
    print(f"❌ '{input_file}' 파일을 찾을 수 없습니다.")
except Exception as e:
    print(f"❌ 오류 발생: {e}")

⏳ 1. 연령대별 데이터를 분석 중입니다...

📊 [연령대별 가중치 산출 결과]
   연령대  발생건수  Age_Weight
60-69세 36593      1.0000
50-59세 20157      0.5508
40-49세  8651      0.2364
30-39세  3700      0.1011
20-29세  2061      0.0563
   10대  1249      0.0341

💾 저장 완료: F:\Download\Step5_Age_Weight.xlsx



In [11]:
import pandas as pd

# 1. 파일 경로 지정 (이전 스텝에서 컬럼 매핑까지 완료된 파일 사용)
input_file = r'F:\Download\Step5_Rawdata_Mapped_to_Columns.xlsx'
output_file = r'F:\Download\Step5_Coverage_Occurrence_Weight.xlsx'

try:
    print("⏳ 2. 보험보장 사고 발생 비중을 분석 중입니다...")
    df = pd.read_excel(input_file, engine='openpyxl')
    
    # 매핑된 보장 컬럼(텍스트) 가져오기
    # 예: "통원의료비_상해외래, 3대비급여_MRI/MRA" -> 리스트로 분리 후 개별 카운트
    # 결측치 제거 후 쉼표 기준으로 쪼개기
    coverage_series = df['필요_보장_컬럼(Text)'].dropna().astype(str)
    
    # 텍스트를 리스트로 변환 후 전개(explode)하여 모든 필요 보장 항목을 1차원 나열
    coverage_list = coverage_series.apply(lambda x: [item.strip() for item in x.split(',') if item.strip()])
    exploded_coverage = coverage_list.explode()
    
    # 보장 항목별 빈도수 집계
    coverage_counts = exploded_coverage.value_counts().reset_index()
    coverage_counts.columns = ['보험보장_항목', '필요_발생건수']
    
    # 가중치 계산 (가장 많이 필요한 보장 항목을 1.0으로 설정)
    max_req_count = coverage_counts['필요_발생건수'].max()
    coverage_counts['Occurrence_Weight'] = (coverage_counts['필요_발생건수'] / max_req_count).round(4)
    
    # 결과 출력 및 저장
    print("\n📊 [보험보장 사고 발생 비중별 가중치]")
    print(coverage_counts.to_string(index=False))
    
    coverage_counts.to_excel(output_file, index=False, engine='openpyxl')
    print(f"\n💾 저장 완료: {output_file}\n")

except FileNotFoundError:
    print(f"❌ '{input_file}' 파일을 찾을 수 없습니다. (먼저 매핑 코드를 실행했는지 확인해주세요.)")
except Exception as e:
    print(f"❌ 오류 발생: {e}")

⏳ 2. 보험보장 사고 발생 비중을 분석 중입니다...

📊 [보험보장 사고 발생 비중별 가중치]
             보험보장_항목  필요_발생건수  Occurrence_Weight
          입원의료비_질병입원    56636             1.0000
  특수리스크_해외질병의료비(USD)    56636             1.0000
사망및장해_질병사망및80%이상후유장해    37704             0.6657
          통원의료비_질병외래    18932             0.3343
          입원의료비_상해입원     2173             0.0384
  특수리스크_해외상해의료비(USD)     2173             0.0384
          사망및장해_상해사망     2081             0.0367
        사망및장해_상해후유장해     2081             0.0367
         특수리스크_구원자비용     2081             0.0367
          통원의료비_상해외래     1119             0.0198
 3대비급여_도수·체외충격파·증식치료     1027             0.0181
       3대비급여_MRI/MRA     1027             0.0181
        3대비급여_비급여주사료     1027             0.0181

💾 저장 완료: F:\Download\Step5_Coverage_Occurrence_Weight.xlsx



In [12]:
import pandas as pd
import numpy as np

# 1. 파일 경로 지정 (Step3 통합 보험 데이터)
input_file = r'F:\Download\Step3_통합_보험데이터_최종_1개월_월환산완료.csv'
output_file = r'F:\Download\Step5_Coverage_Cost_Weight.xlsx'

try:
    print("⏳ 3. 보장별 평균 비용을 분석 중입니다...")
    try:
        df = pd.read_csv(input_file)
    except UnicodeDecodeError:
        df = pd.read_csv(input_file, encoding='cp949')
        
    # 분석에서 제외할 기본 정보 컬럼
    basic_cols = ['국가', '상품플랜', '가입나이', '성별', '최종보험료']
    coverage_cols = [col for col in df.columns if col not in basic_cols]
    
    summary_list = []
    
    # 보장항목별 평균 금액 계산 (0원인 플랜 제외)
    for col in coverage_cols:
        numeric_series = pd.to_numeric(df[col], errors='coerce').fillna(0)
        active_coverage = numeric_series[numeric_series > 0]
        
        avg_amt = active_coverage.mean() if len(active_coverage) > 0 else 0
        summary_list.append({'보험보장_항목': col, '평균보장비용': avg_amt})
        
    weight_df = pd.DataFrame(summary_list)
    
    # 평균비용이 0인 항목 제외
    weight_df = weight_df[weight_df['평균보장비용'] > 0].copy()
    
    # 가중치 정규화 (로그 변환 -> Min-Max Scaling -> 0.1~1.0 보정)
    weight_df['Log_Cost'] = np.log1p(weight_df['평균보장비용'])
    max_log = weight_df['Log_Cost'].max()
    min_log = weight_df['Log_Cost'].min()
    
    weight_df['Cost_Weight'] = (
        (weight_df['Log_Cost'] - min_log) / (max_log - min_log) * 0.9 + 0.1
    ).round(4)
    
    # 정렬 및 정리
    weight_df = weight_df.sort_values(by='Cost_Weight', ascending=False).reset_index(drop=True)
    final_output = weight_df[['보험보장_항목', '평균보장비용', 'Cost_Weight']]
    
    # 결과 출력 및 저장
    print("\n📊 [보험보장별 비용 가중치]")
    print(final_output.to_string(index=False))
    
    final_output.to_excel(output_file, index=False, engine='openpyxl')
    print(f"\n💾 저장 완료: {output_file}\n")

except FileNotFoundError:
    print(f"❌ '{input_file}' 파일을 찾을 수 없습니다.")
except Exception as e:
    print(f"❌ 오류 발생: {e}")

⏳ 3. 보장별 평균 비용을 분석 중입니다...

📊 [보험보장별 비용 가중치]
             보험보장_항목       평균보장비용  Cost_Weight
          사망및장해_상해사망 9.000000e+07       1.0000
        사망및장해_상해후유장해 9.000000e+07       1.0000
        입원의료비_상해질병통합 5.000000e+07       0.9294
  특수리스크_해외상해의료비(USD) 2.025000e+07       0.8209
사망및장해_질병사망및80%이상후유장해 1.875000e+07       0.8117
           일상손해_배상책임 1.500000e+07       0.7849
  특수리스크_해외질병의료비(USD) 1.350000e+07       0.7722
      입원의료비_상해급여(국내) 1.000000e+07       0.7362
      입원의료비_질병급여(국내) 1.000000e+07       0.7362
         특수리스크_구원자비용 1.000000e+07       0.7362
          3대비급여_통합한도 9.000000e+06       0.7235
          입원의료비_상해입원 5.000000e+06       0.6529
          입원의료비_질병입원 5.000000e+06       0.6529
 3대비급여_도수·체외충격파·증식치료 3.500000e+06       0.6101
       3대비급여_MRI/MRA 3.000000e+06       0.5916
        3대비급여_비급여주사료 2.500000e+06       0.5697
    일상손해_휴대품손해(분실제외) 4.666667e+05       0.3682
          통원의료비_상해외래 2.500000e+05       0.2932
          통원의료비_질병외래 2.500000e+05       0.2932
       특수리스크_항공